In [ ]:
%load_ext autoreload
%autoreload 2

# EBiEOT on Colored MNIST (2 → 3)

- **MLP (flattened):** `conf/experiment/egeot_colored_mnist.yaml` — sections below through §6.
- **CNN (image space):** `conf/experiment/egeot_colored_mnist_cnn_{vanilla,nonlocal,unet}.yaml` — §7; train via `python scripts/train_colored_mnist.py experiment=egeot_colored_mnist_cnn_vanilla`.

Data helpers: `src/utils/datasets/colored_mnist.py`.

## 1. Imports

In [ ]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import os

import torch
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from torch import optim
from tqdm import tqdm

from src.utils.datasets.colored_mnist import flatten_images
from src.utils.experiment.logging_utils import init_comet_from_cfg, is_empty_logger_cfg
from src.utils.training import CometExperiment
from src.utils.training.helpers import compute_loss, update_average
from src.utils.plotting.distributions import plot_PCA
from src.utils.plotting.images import plot_image_grids_vertical, plot_transport_pairs
from src.utils.samplers.colored_mnist import (
    build_colored_mnist_image_samplers,
    build_colored_mnist_samplers,
)
from src.utils.core.seed import set_seed


def hydra_experiment(display_name: str) -> str:
    """Map hyphenated preset names to Hydra experiment config names."""
    aliases = {
        "egeot-colored-mnist": "egeot_colored_mnist",
    }
    return aliases.get(display_name, display_name.replace("-", "_"))


In [ ]:
device = torch.device(
    f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
)
device

In [ ]:
torch.set_default_device(device)
dtype = torch.float32
torch.set_default_dtype(dtype)

## 2. Config

In [ ]:
# Papermill: EXPERIMENT selects conf/experiment/<name>.yaml; OVERRIDES are Hydra CLI-style strings.
EXPERIMENT = "egeot-colored-mnist"
OVERRIDES: list[str] = [
    # "logger=comet",  # enable Comet ML (conf/logger/comet.yaml)
    # "train.steps_to=1000",
    # "dataset.P_XY_paired=200",
]

CONF_DIR = os.path.abspath(str(REPO_ROOT / "conf"))
with initialize_config_dir(version_base=None, config_dir=CONF_DIR):
    cfg = compose(
        config_name="config",
        overrides=[f"experiment={hydra_experiment(EXPERIMENT)}", *OVERRIDES],
    )
seed = int(cfg.seed) if cfg.get("seed") is not None else int(cfg.train.seed)
set_seed(seed)
print(OmegaConf.to_yaml(cfg.train))


Config is composed from `conf/experiment/egeot_colored_mnist.yaml` via Hydra `compose` (see §2). Override dataset sizes, steps, or MLP widths with `OVERRIDES`. Add `"logger=comet"` to `OVERRIDES` to log metrics and transport figures to Comet ML.

CLI equivalent:

```bash
python scripts/train.py experiment=egeot_colored_mnist
```

*(Colored MNIST training is notebook-only today; `scripts/train.py` still wires Swiss-roll samplers.)*

## 3. Model and data

In [ ]:
ds = cfg.dataset
IMG_SIZE = int(ds.img_size)
CHANNELS = int(ds.channels)

model = build_neural_model(cfg, device)
usd_sampler, utd_sampler, pd_sampler, X_paired_img, Y_paired_img = (
    build_colored_mnist_samplers(cfg, device)
)
X_paired_train = flatten_images(X_paired_img)
Y_paired_train = flatten_images(Y_paired_img)

model_copy = None
if cfg.train.ema_update:
    model_copy = build_neural_model(cfg, device)
    model_copy.load_state_dict(model.state_dict())

print(
    f"P_XY: {X_paired_train.shape}; Q_X: {usd_sampler.dataset.shape}; "
    f"R_Y: {utd_sampler.dataset.shape}"
)

In [ ]:
plot_image_grids_vertical(
    [X_paired_img[:16], Y_paired_img[:16]],
    ["paired source (digit 2)", "paired target (digit 3)"],
    nrow=8,
)

## 4. Optimizers

In [ ]:
def _adam(params, opt_cfg):
    return optim.Adam(
        params,
        lr=float(opt_cfg.lr),
        betas=tuple(float(b) for b in opt_cfg.betas),
    )


D_opt_unpaired = _adam(model.potential.parameters(), cfg.train.optimizer.unpaired)
D_opt_paired = _adam(model.cost.parameters(), cfg.train.optimizer.paired)

train_cfg = cfg.train
EXP_NAME = (
    f"EBiEOT-ColoredMNIST-{EXPERIMENT}-"
    f"P{ds.P_XY_paired}_Q{ds.Q_X_unpaired}_R{ds.R_Y_unpaired}"
)
OUTPUT_PATH = REPO_ROOT / "checkpoints" / EXP_NAME
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
OmegaConf.save(cfg, OUTPUT_PATH / "config.yaml")

experiment = None
comet_experiment = None
if not is_empty_logger_cfg(cfg.get("logger")):
    experiment = init_comet_from_cfg(
        cfg.logger, full_cfg=cfg, name=EXP_NAME, output_dir=OUTPUT_PATH
    )
    if experiment is not None:
        comet_experiment = CometExperiment(experiment)


### Image-space EBiEOT-NN

Energy is defined on pixels via CNN (or flattened MLP) cost and potential. Unpaired \(Y\) samples are refined with Langevin dynamics and a replay buffer so the conditional sampler tracks the learned transport.

## 5. Training

In [ ]:
eval_indices = torch.arange(min(8, X_paired_train.shape[0]), device=device)
eval_x = X_paired_train[eval_indices]

for step in tqdm(range(int(train_cfg.steps_from), int(train_cfg.steps_to))):
    D_opt_unpaired.zero_grad()
    X = usd_sampler.sample(int(train_cfg.unpaired_batch_size))
    Y = utd_sampler.sample(int(train_cfg.unpaired_batch_size))
    out_u = model.compute_unpaired_loss(X, Y)
    loss_u = out_u["loss"]

    D_opt_paired.zero_grad()
    X_p, Y_p = pd_sampler.sample(int(train_cfg.paired_batch_size))
    out_p = model.compute_paired_loss(X_p, Y_p)
    loss_p = out_p["loss"]

    loss = loss_u + loss_p
    loss.backward()
    D_opt_unpaired.step()
    D_opt_paired.step()

    if train_cfg.ema_update and model_copy is not None:
        update_average(model_copy, model, 0.99)

    if experiment is not None:
        experiment.log_metrics(
            {
                "loss": loss.item(),
                "unpaired_loss": loss_u.item(),
                "paired_loss": loss_p.item(),
            },
            step=step,
        )

    if int(train_cfg.plot_every) > 0 and step % int(train_cfg.plot_every) == 0:
        plot_model = model_copy if model_copy is not None else model
        with torch.no_grad():
            moved = plot_model(eval_x)
        plot_transport_pairs(
            eval_x,
            Y_paired_train[eval_indices],
            moved,
            channels=CHANNELS,
            img_size=IMG_SIZE,
            experiment=comet_experiment,
            step=step,
        )
        torch.save(model.state_dict(), OUTPUT_PATH / f"D_{step}.pt")

torch.save(model.state_dict(), OUTPUT_PATH / f"D_{train_cfg.steps_to}.pt")
torch.save(D_opt_paired.state_dict(), OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_to}.pt")
torch.save(
    D_opt_unpaired.state_dict(), OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_to}.pt"
)

if experiment is not None:
    experiment.end()


## 6. Evaluation plots

In [ ]:
plot_model = model_copy if model_copy is not None else model
n_eval = min(64, X_paired_train.shape[0])
idx = torch.arange(n_eval, device=device)
x_eval = X_paired_train[idx]
y_eval = Y_paired_train[idx]

with torch.no_grad():
    y_pred = plot_model(x_eval)

plot_transport_pairs(
    x_eval,
    y_eval,
    y_pred,
    num_show=8,
    channels=CHANNELS,
    img_size=IMG_SIZE,
)

plot_PCA(
    plot_model,
    usd_sampler.tensor[:n_eval],
    utd_sampler.tensor[:n_eval],
    x_eval,
    y_eval,
)

print(
    "paired eval loss:",
    compute_loss(plot_model, x_eval, y_eval, x_eval, y_eval),
)

## 7. CNN experiments (image tensors)

Compose a CNN preset and use image samplers (no `flatten_images`). CLI: `scripts/train_colored_mnist.py`.

In [ ]:
CNN_EXPERIMENT = "egeot_colored_mnist_cnn_vanilla"  # or _nonlocal, _unet

with initialize_config_dir(version_base=None, config_dir=str(REPO_ROOT / "conf")):
    cfg_cnn = compose(
        config_name="config",
        overrides=[f"experiment={CNN_EXPERIMENT}", "train.steps_to=10"],
    )

OmegaConf.resolve(cfg_cnn)
model_cnn = build_neural_model(cfg_cnn, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
usd_cnn, utd_cnn, pd_cnn, x_paired_img, y_paired_img = build_colored_mnist_image_samplers(
    cfg_cnn, next(model_cnn.parameters()).device
)
print("CNN model:", type(model_cnn.cost).__name__, "| paired images:", tuple(x_paired_img.shape))